In [0]:
import requests, json, time
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ---------- 3) RemoteOK ----------
def fetch_remoteok():
    r = requests.get("https://remoteok.com/api",
                      headers={"User-Agent": "job-copilot-capstone"}, timeout=20)
    r.raise_for_status()
    data = r.json()
    return [d for d in data if isinstance(d, dict) and "id" in d]

remoteok_raw = fetch_remoteok()

# snimi sirove JSON-e u Volume (audit trag + fallback ako transformacija pukne)
dbutils.fs.put("/Volumes/job_copilot/raw/landing/remoteok.json", json.dumps(remoteok_raw), overwrite=True)

In [0]:
schema = StructType([
    StructField("source", StringType()),
    StructField("external_id", StringType()),
    StructField("title", StringType()),
    StructField("company", StringType()),
    StructField("location", StringType()),
    StructField("description", StringType()),
    StructField("url", StringType()),
    StructField("salary_min", DoubleType()),
    StructField("salary_max", DoubleType()),
    StructField("remote", BooleanType()),
    StructField("posted_at", StringType()),
])


def normalize_remoteok(d):
    return {
        "source": "remoteok", "external_id": str(d.get("id")),
        "title": d.get("position"), "company": d.get("company"),
        "location": d.get("location") or "Remote",
        "description": d.get("description"), "url": d.get("url"),
        "salary_min": d.get("salary_min"), "salary_max": d.get("salary_max"),
        "remote": True, "posted_at": d.get("date"),
    }

rows = [normalize_remoteok(d) for d in remoteok_raw]

df = spark.createDataFrame(rows, schema=schema)

df_clean = (
    df
    .filter(
        F.col("title").isNotNull() &
        F.col("description").isNotNull() &
        F.col("external_id").isNotNull()
    )
    .withColumn(
        "description",
        F.regexp_replace("description", "<[^>]*>", "")
    )
    .withColumn(
        "job_id",
        F.concat_ws("_", "source", "external_id")
    )
    .withColumn("ingested_at", F.current_timestamp())
    .dropDuplicates(["job_id"])
)

(df_clean.write.mode("append")
    .format("delta")
    .saveAsTable("job_copilot.clean.job_postings"))

print(df_clean.count(), "novih oglasa upisano")

In [0]:
import psycopg2
from psycopg2.extras import execute_values
lb_host = "ep-frosty-bar-d8f2o7dt.database.us-east-2.cloud.databricks.com"
lb_port = 5432
lb_db = "databricks_postgres"
lb_user = "gogsbogs"

lb_password = dbutils.secrets.get(
    scope="job-copilot",
    key="lakebase-password"
).strip('\x00')
conn = psycopg2.connect(
    host=lb_host,
    port=lb_port,
    dbname=lb_db,
    user=lb_user,
    password=lb_password,
    sslmode="require"
)
print("Successfully connected to Lakebase!")

In [0]:

rows = df_clean.select("source","external_id","title","company","location",
                        "description","url","salary_min","salary_max","remote","posted_at").collect()

records = [(f"{r.source}_{r.external_id}", r.source, r.title, r.company, r.location,
            r.description, r.url, r.salary_min, r.salary_max, r.remote, r.posted_at) for r in rows]

with conn.cursor() as cur:
    execute_values(cur, """
        INSERT INTO job_postings (job_id, source, title, company, location, description, url,
                                   salary_min, salary_max, remote, posted_at)
        VALUES %s
        ON CONFLICT (job_id) DO UPDATE SET synced_at = now()
    """, records)
conn.commit()

Celija ispod se samo jednom pozove da se kreira tabela. Error baca ponovni poziv. Preskociti pri pokretanju vise puta

Unstructured data + embeddings + Vector Search

Ovde obrađuješ nestrukturirani tekst (opisi poslova) i praviš semantičku pretragu.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS job_copilot.vector;
CREATE TABLE job_copilot.vector.job_postings_src (
    job_id STRING,
    title STRING,
    company STRING,
    description STRING,
    combined_text STRING   -- title + qualifications + description spojeno, za embedding
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
from pyspark.sql import functions as F

df_vec = (df_clean
    .withColumn("job_id", F.concat_ws("_", "source", "external_id"))
    .withColumn("combined_text", F.concat_ws(" | ", F.col("title"), F.col("company"), F.col("description")))
    .select("job_id", "title", "company", "description", "combined_text"))

df_vec.write.mode("append").format("delta").saveAsTable("job_copilot.vector.job_postings_src")

In [0]:
from pyspark.sql import functions as F

df_vec = (df_clean
    .withColumn("job_id", F.concat_ws("_", "source", "external_id"))
    .withColumn("combined_text", F.concat_ws(" | ", F.col("title"), F.col("company"), F.col("description")))
    .select("job_id", "title", "company", "description", "combined_text"))

df_vec.write.mode("append").format("delta").saveAsTable("job_copilot.vector.job_postings_src")
df_vec.head(1)

In [0]:
%sql
ALTER TABLE job_copilot.vector.job_postings_src
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

Kod u celiji ispod se izvrsava/interpretira samo jednom. Svaki naredni put baca izuzetak jer resursi vec postoje.

Kreiranje Vector Search endpoint-a i indeksa

U notebook-u (ili preko UI-ja: Compute → Vector Search):

In [0]:
%pip install databricks-vectorsearch
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
vsc.create_endpoint(name="job-copilot-endpoint", endpoint_type="STANDARD")

vsc = VectorSearchClient()
endpoint = vsc.get_endpoint("job-copilot-endpoint")
print(endpoint)
vsc.list_indexes("job-copilot-endpoint")
index = vsc.create_delta_sync_index(
endpoint_name="job-copilot-endpoint",
source_table_name="job_copilot.vector.job_postings_src",
index_name="job_copilot.vector.job_postings_index",
pipeline_type="TRIGGERED",
primary_key="job_id",
embedding_source_column="combined_text",
embedding_model_endpoint_name="databricks-bge-large-en"  # ugrađeni Databricks FM endpoint
)

!!!! TODO !!!!!!
Isto uradi i za profil korisnika (resume/skills) — opciono ali preporučeno

Ako želiš da agent poredi profil korisnika sa oglasima semantički (a ne samo SQL filterima), napravi i tabelu job_copilot.vector.user_profiles_src sa kolonom profile_text (spojen resume + skills), pa isti postupak — poseban indeks ili isti endpoint.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
endpoint = vsc.get_endpoint("job-copilot-endpoint")
print(endpoint)
vsc.list_indexes("job-copilot-endpoint")
index = vsc.get_index(
    endpoint_name="job-copilot-endpoint",
    index_name="job_copilot.vector.job_postings_index"
)

Pretraga iz koda (koristi agent tool kasnije)